# 02 Procesamiento de texto, recuperación secuencial e indexación

## Objetivo

Implementar y comparar dos mecanismos de recuperación de información —búsqueda secuencial e índice invertido— sobre colecciones de libros del Proyecto Gutenberg, investigando experimentalmente su rendimiento y escalabilidad.

### Parte A. Corpus pequeño

**1. Preparación del corpus**

Utilice los 18 documentos de la colección Gutenberg disponible en NLTK.

1. Cargue los textos originales.
2. Identifique el número de documentos y sus tamaños.
3. Implemente tokenización y normalización: conversión a minúsculas, tratamiento de puntuación y caracteres especiales.

Seleccione y documente una configuración de preprocesamiento que se utilizará consistentemente en todos los experimentos.


**Nombre:** Harry Guajan  
**Fecha:** 23 de octubre de 2026

In [1]:
import re
from collections import Counter
from time import perf_counter_ns

import nltk
import pandas as pd
from IPython.display import display

try:
    from nltk.corpus import gutenberg
    document_ids = gutenberg.fileids()
    if len(document_ids) != 18:
        raise LookupError("La colección Gutenberg de NLTK no está disponible.")
except LookupError:
    nltk.download("gutenberg", quiet=True)
    from nltk.corpus import gutenberg
    document_ids = gutenberg.fileids()

# Que se considera un token valido
token_pattern = re.compile(r"[a-z0-9]+(?:'[a-z0-9]+)?")
# Normalizacion de texto y tokenizacion 
def normalize(text):
    return token_pattern.findall(text.lower())

original_documents = {doc_id: gutenberg.raw(doc_id) for doc_id in document_ids}
tokenized_documents = {
    doc_id: normalize(text) for doc_id, text in original_documents.items()
}

document_summary = pd.DataFrame([
    {
        "document_id": doc_id,
        "original_characters": len(original_documents[doc_id]),
        "tokens": len(tokenized_documents[doc_id]),
    }
    for doc_id in document_ids
])

print(f"Documentos cargados: {len(document_ids)}")
print(f"Tokens totales: {document_summary['tokens'].sum():,}")
display(document_summary)


Documentos cargados: 18
Tokens totales: 2,182,070


,document_id,original_characters,tokens
0,austen-emma.txt,887071,160999
1,austen-persuasion.txt,466292,83660
2,austen-sense.txt,673022,119965
3,bible-kjv.txt,4332554,852256
4,blake-poems.txt,38153,6839
5,bryant-stories.txt,249439,46321
6,burgess-busterbrown.txt,84663,15969
7,carroll-alice.txt,144395,26688
8,chesterton-ball.txt,457450,82252
9,chesterton-brown.txt,406629,72391


**2. Búsqueda secuencial**

Implemente una función que reciba un término de consulta y devuelva los identificadores de todos los documentos que lo contienen con una búsqueda secuencial.

La implementación deberá utilizar obligatoriamente dos lazos `for` anidados: el primero recorre los documentos y el segundo los tokens de cada documento.

No es necesario utilizar índices invertidos, motores de búsqueda ni estructuras auxiliares que almacenen anticipadamente los documentos asociados a cada término.

Ejecute consultas con términos frecuentes, poco frecuentes e inexistentes.

In [2]:
def sequential_search(query):
    #Devuelve los identificadores de los documentos que contienen el término
    query_tokens = normalize(query)

    if len(query_tokens) != 1:
        raise ValueError("La búsqueda secuencial recibe exactamente un término.")

    term = query_tokens[0]
    matches = []

    for document_id, tokens in tokenized_documents.items():
        for token in tokens:
            if token == term:
                matches.append(document_id)
                break

    return matches

# Frecuencia total de apariciones de cada término.
term_counts = Counter(
    token
    for tokens in tokenized_documents.values()
    for token in tokens
)

# Número de documentos distintos en los que aparece cada término.
document_frequency = Counter(
    token
    for tokens in tokenized_documents.values()
    for token in set(tokens)
)

# Consulta frecuente: término presente en más documentos.
#most_common(1): obtiene el término más frecuente por número de documentos.
#[0]: obtiene la primera tupla de resultados.
#[0]: obtiene el término de esa tupla, descartando su frecuencia.

frequent_term = document_frequency.most_common(1)[0][0]

# Consulta poco frecuente: término presente en un solo documento,
# si existe alguno que cumpla los filtros.
rare_candidates = [
    term
    for term, frequency in document_frequency.items()
    if frequency == 1 and term.isalpha() and len(term) > 2
]

if rare_candidates:
    infrequent_term = min(
        rare_candidates,
        key=lambda term: (term_counts[term], term)
    )
else:
    # Si no hay términos presentes en un solo documento,
    # seleccionar el de menor frecuencia documental.
    infrequent_term = min(
        (
            term for term in document_frequency
            if term.isalpha() and len(term) > 2
        ),
        key=lambda term: (
            document_frequency[term],
            term_counts[term],
            term
        )
    )

# Consulta inexistente: un solo token que no aparece en el corpus.
missing_term = "Insaneador"

# Ejecutar las consultas y mostrar sus resultados.
queries = [
    ("frecuente", frequent_term),
    ("poco frecuente", infrequent_term),
    ("inexistente", missing_term)
]

sequential_examples = pd.DataFrame([
    {
        "query": label,
        "term": term,
        "document_frequency": document_frequency.get(term, 0),
        "documents": sequential_search(term)
    }
    for label, term in queries
])

display(sequential_examples)

,query,term,document_frequency,documents
0,frecuente,men,18,"[austen-emma.txt, austen-persuasion.txt, auste..."
1,poco frecuente,abaddon,1,[bible-kjv.txt]
2,inexistente,Insaneador,0,[]


**3. Medición de tiempos de ejecución**

Utilice `time.perf_counter_ns()` o una herramienta de medición equivalente para obtener el tiempo de ejecución de las consultas.

Cada medición deberá corresponder a **30 ejecuciones de la misma consulta**. Y se debe calcular la media aritmética.

Registre los resultados en milisegundos. No incluya la carga de archivos ni la tokenización dentro del tiempo de consulta.

In [3]:
def mean_query_time(query, search_function, repetitions=30):
    elapsed = []
    for _ in range(repetitions):
        start = perf_counter_ns()
        search_function(query)
        elapsed.append(perf_counter_ns() - start)
    return sum(elapsed) / repetitions / 1_000_000

sequential_timing = pd.DataFrame([
    {
        "query_type": label,
        "term": term,
        "mean_time_ms": mean_query_time(term, sequential_search),
        "repetitions": 30,
    }
    for label, term in [
        ("frecuente", frequent_term),
        ("poco frecuente", infrequent_term),
        ("inexistente", missing_term),
    ]
])
display(sequential_timing)


,query_type,term,mean_time_ms,repetitions
0,frecuente,men,2.848140,30
1,poco frecuente,abaddon,55.587453,30
2,inexistente,Insaneador,51.286900,30


#### Observacion
Esperaba que los tiempos fueran similares, pero tiene lógica que el tiempo para encontrar el término frecuente sea menor, aunque esto no siempre tiene que ser así. Puede que men aparezca más veces, pero que se encuentre después de la primera aparición de abaddon. Por lo tanto, esto no comprueba que los términos frecuentes sean necesariamente más rápidos.

Por otro lado, entre el término poco frecuente y el inexistente hay poca diferencia, ya que el algoritmo recorre casi todos los documentos en ambos casos. Además, cuando el término no existe, debe recorrerlos todos.

**4. Construcción del índice invertido**

Implemente un índice invertido que contenga el vocabulario de la colección y las listas de documentos asociados a cada término.

Inspeccione las listas de postings y calcule la frecuencia documental (DF) de varios términos.

Implemente consultas de un término y consultas booleanas con AND, OR y NOT.

In [ ]:
# Construcción del índice invertido
inverted_index = {}

for document_id, tokens in tokenized_documents.items():
    for token in set(tokens):
        inverted_index.setdefault(token, set()).add(document_id)

all_documents = set(document_ids)

# Consultas
def indexed_search(query):
    tokens = normalize(query)
    if len(tokens) != 1:
        raise ValueError("La consulta debe contener exactamente un término.")
    return sorted(inverted_index.get(tokens[0], set()))

def boolean_and(left, right):
    return sorted(inverted_index.get(left, set()) & inverted_index.get(right, set()))

def boolean_or(left, right):
    return sorted(inverted_index.get(left, set()) | inverted_index.get(right, set()))

def boolean_not(term):
    return sorted(all_documents - inverted_index.get(term, set()))


# Inspección de postings y frecuencia documental (DF)
postings_terms = dict.fromkeys([frequent_term, infrequent_term, "the"])

postings = pd.DataFrame([
    {
        "term": term,
        "df": len(docs := inverted_index.get(term, set())),
        "postings": sorted(docs)
    }
    for term in postings_terms
])

display(postings)


# Ejemplos de consultas booleanas
boolean_examples = pd.DataFrame([
    {"query": f"{frequent_term} AND {infrequent_term}",
     "documents": boolean_and(frequent_term, infrequent_term)},
    {"query": f"{frequent_term} OR {infrequent_term}",
     "documents": boolean_or(frequent_term, infrequent_term)},
    {"query": f"NOT {infrequent_term}",
     "documents": boolean_not(infrequent_term)}
])

display(boolean_examples)

,term,df,postings
0,men,18,"[austen-emma.txt, austen-persuasion.txt, auste..."
1,abaddon,1,[bible-kjv.txt]
2,the,18,"[austen-emma.txt, austen-persuasion.txt, auste..."


,query,documents
0,men AND abaddon,[bible-kjv.txt]
1,men OR abaddon,"[austen-emma.txt, austen-persuasion.txt, auste..."
2,NOT abaddon,"[austen-emma.txt, austen-persuasion.txt, auste..."


**5. Comparación**

Ejecute las mismas consultas con ambos métodos.

Compruebe que recuperan exactamente los mismos identificadores de documentos y mida el tiempo medio de 30 ejecuciones para cada consulta y método.

In [ ]:
comparison_queries = [
    ("frecuente", frequent_term),
    ("poco frecuente", infrequent_term),
    ("inexistente", missing_term)
]

comparison_results = []

for query_type, term in comparison_queries:
    # Medir el tiempo medio de 30 ejecuciones
    sequential_time = mean_query_time(term, sequential_search, 30)
    indexed_time = mean_query_time(term, indexed_search, 30)

    # Obtener los documentos recuperados
    sequential_documents = sequential_search(term)

    comparison_results.append({
        "query_type": query_type,
        "term": term,
        "documents": sequential_documents,
        "sequential_time_ms": sequential_time,
        "indexed_time_ms": indexed_time,
        "speedup": sequential_time / indexed_time
    })

comparison_df = pd.DataFrame(comparison_results)

display(comparison_df)

,query_type,term,documents,sequential_time_ms,indexed_time_ms,speedup
0,frecuente,men,"[austen-emma.txt, austen-persuasion.txt, auste...",1.303673,0.003037,429.310648
1,poco frecuente,abaddon,[bible-kjv.txt],29.967987,0.002400,12486.661111
2,inexistente,Insaneador,[],25.486357,0.002700,9439.391358


##### Observaciones de las comparaciones
Los resultados muestran una gran diferencia entre los métodos. Como vimos en clase, la búsqueda secuencial tiene una complejidad de \(O(N)\), mientras que el índice invertido tiene una complejidad promedio de \(O(1)\) para localizar los términos. Para el término men, la búsqueda secuencial tardó 1,3 ms y el índice invertido 0,0030 ms, siendo este último aproximadamente 429 veces más rápido. Con abaddon, resultó 12 487 veces más rápido, y con el término inexistente, 9439 veces.

Estas pruebas se realizaron en un equipo con procesador Intel Core i5-1335U de 13.ª generación, 8 GB de RAM y sistema operativo de 64 bits. Los tiempos pueden variar según la carga del sistema y las condiciones de ejecución, pero mantendrán la misma tendencia.

### Parte B. Corpus grande

**6. Descarga de Gutenberg**

Descargue el archivo oficial `txt-files.tar.zip`, de aproximadamente 11 GB, desde:

https://www.gutenberg.org/cache/epub/feeds/

Identifique los archivos TXT, evite contar distintas versiones del mismo libro como documentos independientes y registre la cantidad de documentos válidos.

In [13]:
# Parte B: Descarga ya realizada, extracción, identificación y recuento
# de documentos de Project Gutenberg

from pathlib import Path
from collections import defaultdict
import zipfile
import tarfile
import shutil
import re
import pandas as pd
from IPython.display import display

# 1. Configuración de rutas
base_dir = Path(
    r"C:\Users\harry\Desktop\Universidad\Semestre 2026-B\Recuperaicon de la Informacion\SRCD2"
)
archive_path = base_dir / "txt-files.tar.zip"
extract_dir = base_dir / "gutenberg_txt"
inventory_path = base_dir / "gutenberg_inventory.csv"

extract_dir.mkdir(parents=True, exist_ok=True)

if not archive_path.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo ZIP: {archive_path}"
    )

print(f"Archivo ZIP: {archive_path}")
print(f"Tamaño: {archive_path.stat().st_size / (1024**3):.2f} GB")

# 2. Extraer el corpus si todavía no existen archivos TXT
existing_txt = list(extract_dir.glob("*.txt"))

if existing_txt:
    print(
        f"Ya existen {len(existing_txt)} archivos TXT. "
        "Se reutilizarán y se buscarán archivos faltantes."
    )

else:
    print("Iniciando extracción del corpus...")

    with zipfile.ZipFile(archive_path, "r") as zip_ref:
        tar_members = [
            name for name in zip_ref.namelist()
            if name.lower().endswith(".tar")
        ]

        if tar_members:
            # Extraer el TAR temporalmente
            tar_path = base_dir / Path(tar_members[0]).name

            if not tar_path.exists():
                with zip_ref.open(tar_members[0]) as source:
                    with open(tar_path, "wb") as target:
                        shutil.copyfileobj(source, target)

            with tarfile.open(tar_path, "r:*") as tar_ref:
                for member in tar_ref:
                    if not member.isfile():
                        continue

                    if not member.name.lower().endswith(".txt"):
                        continue

                    filename = Path(member.name).name

                    if not filename:
                        continue

                    destination = extract_dir / filename

                    if destination.exists():
                        continue

                    source = tar_ref.extractfile(member)

                    if source is not None:
                        with source, open(destination, "wb") as target:
                            shutil.copyfileobj(source, target)

            print("Extracción del TAR completada.")

        else:
            # Si el ZIP contiene directamente los TXT
            for member in zip_ref.infolist():
                if member.is_dir():
                    continue

                if not member.filename.lower().endswith(".txt"):
                    continue

                filename = Path(member.filename).name

                if not filename:
                    continue

                destination = extract_dir / filename

                if destination.exists():
                    continue

                with zip_ref.open(member) as source:
                    with open(destination, "wb") as target:
                        shutil.copyfileobj(source, target)

            print("Extracción directa del ZIP completada.")

# 3. Identificar los archivos TXT
txt_files = list(extract_dir.glob("*.txt"))

# 4. Agrupar los archivos por ID de Gutenberg
documents_by_id = defaultdict(list)

for path in txt_files:
    match = re.fullmatch(
        r"pg(\d+)",
        path.stem,
        flags=re.IGNORECASE
    )

    if match:
        doc_id = match.group(1)
        documents_by_id[doc_id].append(path)

# 5. Excluir archivos vacíos y seleccionar un archivo por ID
valid_documents = {}
empty_files = []

for doc_id, paths in documents_by_id.items():
    nonempty_paths = [
        path for path in paths
        if path.stat().st_size > 0
    ]

    if not nonempty_paths:
        empty_files.extend(paths)
        continue

    # Si existen varios archivos para un mismo ID, elegir uno
    selected_path = sorted(
        nonempty_paths,
        key=lambda path: path.name.lower()
    )[0]

    valid_documents[doc_id] = selected_path

# 6. Crear el inventario de documentos
documents_df = pd.DataFrame([
    {
        "doc_id": int(doc_id),
        "filename": path.name,
        "path": str(path),
        "size_bytes": path.stat().st_size
    }
    for doc_id, path in valid_documents.items()
])

if not documents_df.empty:
    documents_df = (
        documents_df
        .sort_values("doc_id")
        .reset_index(drop=True)
    )

# 7. Guardar el inventario
documents_df.to_csv(
    inventory_path,
    index=False,
    encoding="utf-8"
)

# 8. Mostrar el resumen
print("\n--- RESULTADOS DEL CORPUS ---")
print(f"Archivos TXT encontrados: {len(txt_files)}")
print(f"Archivos con ID reconocible: "
      f"{sum(len(paths) for paths in documents_by_id.values())}")
print(f"IDs únicos identificados: {len(documents_by_id)}")
print(f"Archivos vacíos: {len(empty_files)}")
print(f"Documentos no vacíos seleccionados: {len(valid_documents)}")
print(f"Inventario guardado en: {inventory_path}")

display(documents_df.head(10))

Archivo ZIP: C:\Users\harry\Desktop\Universidad\Semestre 2026-B\Recuperaicon de la Informacion\SRCD2\txt-files.tar.zip
Tamaño: 10.54 GB
Iniciando extracción del corpus...
Extracción del TAR completada.

--- RESULTADOS DEL CORPUS ---
Archivos TXT encontrados: 78107
Archivos con ID reconocible: 78107
IDs únicos identificados: 78107
Archivos vacíos: 0
Documentos no vacíos seleccionados: 78107
Inventario guardado en: C:\Users\harry\Desktop\Universidad\Semestre 2026-B\Recuperaicon de la Informacion\SRCD2\gutenberg_inventory.csv


,doc_id,filename,path,size_bytes
0,1,pg1.txt,C:\Users\harry\Desktop\Universidad\Semestre 20...,30835
1,2,pg2.txt,C:\Users\harry\Desktop\Universidad\Semestre 20...,22770
2,3,pg3.txt,C:\Users\harry\Desktop\Universidad\Semestre 20...,27601
3,4,pg4.txt,C:\Users\harry\Desktop\Universidad\Semestre 20...,21788
4,5,pg5.txt,C:\Users\harry\Desktop\Universidad\Semestre 20...,48326
5,6,pg6.txt,C:\Users\harry\Desktop\Universidad\Semestre 20...,26547
6,7,pg7.txt,C:\Users\harry\Desktop\Universidad\Semestre 20...,21888
7,8,pg8.txt,C:\Users\harry\Desktop\Universidad\Semestre 20...,24107
8,9,pg9.txt,C:\Users\harry\Desktop\Universidad\Semestre 20...,41473
9,10,pg10.txt,C:\Users\harry\Desktop\Universidad\Semestre 20...,4455950


### Consideraciones sobre el corpus de Gutenberg

El corpus se obtiene del archivo oficial `txt-files.tar.zip` de Project Gutenberg. Aunque el archivo comprimido ocupa aproximadamente 11 GB, su contenido extraído puede ocupar alrededor de 50 GB de disco.

Las rutas locales deben definirse mediante variables para facilitar la ejecución en diferentes sistemas. El código genera un inventario de los documentos identificados y lo guarda en formato CSV.



**7. Recuperación sobre el corpus grande**

Reutilice las funciones desarrolladas en la Parte A.

1. Procese los documentos con la misma configuración de normalización.
2. Ejecute las consultas mediante búsqueda secuencial y mida los tiempos.
3. Construya el índice invertido.
4. Repita las consultas utilizando el índice.
5. Compruebe la igualdad de resultados y compare los tiempos.

Trabaje con subconjuntos de la colección de tamaños crecientes, conservando los identificadores originales. Documente los recursos de memoria y almacenamiento utilizados.

Espacio libre: 77.49 GB


URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1032)>

### Parte C. Análisis experimental de escalabilidad

**8. Tiempo de ejecución en función del corpus**

Seleccione al menos cinco tamaños crecientes del corpus grande. Como referencia, pueden utilizarse 50, 100, 250, 500 y 1.000 documentos, ampliándolos si el equipo lo permite.

Para cada tamaño $n$:

1. Ejecute una misma consulta mediante búsqueda secuencial.
2. Repítala 30 veces y calcule el tiempo promedio.
3. Construya el índice invertido correspondiente a ese subconjunto.
4. Ejecute la consulta mediante el índice, también 30 veces.
5. Registre ambos tiempos y el número total de tokens.

Elabore una **gráfica del tiempo medio de ejecución en función del número de documentos**, colocando el número de documentos en el eje X y el tiempo en milisegundos en el eje Y. Represente los dos métodos en la misma gráfica.

**9. Estimación de una función de tiempo**

Utilizando los resultados experimentales, determine una función:

$t=f(n)$

para cada método de recuperación.

Como primera aproximación, utilice un ajuste lineal por mínimos cuadrados:

$t(n)=an+b$

Estime los parámetros $a$ y $b$, reporte el coeficiente de determinación $R^2$ y represente las funciones ajustadas junto a los datos experimentales.

Explique si el modelo lineal representa adecuadamente las mediciones y cómo se relaciona con la complejidad computacional de cada algoritmo.